# SharkByte Colab: connected 50-qubit state, up to 4,294,967,296 active amplitudes

This notebook selects the largest included GHZ-family workload that fits Colab's **current free disk plus a 10 GiB reserve**. The 2^32 tier (4,294,967,296 active amplitudes) needs 96 GiB for packed basis/amplitude arrays. A previous run materialized those arrays but reached its 900-second verification limit. This revision gives the top fitting tier 2,400 seconds and does not automatically rerun lower tiers; the 2^31 fallback already has a PASS receipt.

For the selected tier, it writes packed arrays (uint64 basis index and complex128 amplitude), reads and checks every entry against a separate closed-form parity implementation, hashes both full arrays, and deletes the arrays. Only compact receipts, hashes, progress and logs are downloaded. The worker RSS guard is 3.5 GiB; the notebook requires at least 6 GiB available RAM and 10 GiB free disk after the arrays are accounted for. Resource refusal remains a refusal; it is never called a pass.

The source circuit connects all 50 qubits with a 49-edge CNOT chain. Its GHZ backbone is entangled across every nontrivial bipartition; later one-qubit Hadamards preserve its Schmidt coefficients. Cirq independently checks the Clifford measurement invariant. **This is an exhaustive closed-form state materialization for this structured family, not general gate-by-gate simulation or a dense 50-qubit statevector.** A dense complex128 50-qubit vector would require 16 PiB.

Run cells in order. Choose a CPU runtime. Do not run multiple copies at once. The parent monitors and terminates a worker that exceeds memory or wall limits. It attempts only the highest tier that passes preflight, preserving the previous lower-tier PASS rather than repeating it. A mathematical mismatch or unexplained worker failure remains visible in the evidence bundle.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import zipfile

GIB = 1024 ** 3
MIB = 1024 ** 2
MIN_AVAILABLE_RAM = 6 * GIB
DISK_RESERVE = 10 * GIB
WORKER_RSS_LIMIT = int(3.5 * GIB)
WORKER_TIMEOUT = 2700
WORKER_CPU_LIMIT = 2500
CHUNK_ELEMENTS = 1 << 22
RUN_DIR = Path(tempfile.mkdtemp(prefix="sharkbyte-connected-50q-", dir="/content"))
REPO = RUN_DIR / "SharkByte"
RESULTS = RUN_DIR / "results"
SCRATCH = RUN_DIR / "scratch"
PACKAGES = RUN_DIR / "python-packages"
for folder in (RESULTS, SCRATCH, PACKAGES):
    folder.mkdir(parents=True, exist_ok=True)

clone = subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
     "https://github.com/reggaesharkk/SharkByte.git", str(REPO)],
    capture_output=True, text=True)
if clone.returncode:
    raise RuntimeError(clone.stderr or "Could not clone SharkByte main")
SOURCE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
NOTEBOOK_PATH = REPO / "notebooks" / "SharkByte-Colab-Connected-50Q-2B.ipynb"
NOTEBOOK_SHA256 = hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest()

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "--no-warn-script-location", "--target", str(PACKAGES),
     "numpy==2.5.4", "cirq-core==1.6.1"], check=True)
WORKER_ENV = dict(os.environ, MPLBACKEND="Agg", OPENBLAS_NUM_THREADS="1",
                  OMP_NUM_THREADS="1")
old_pythonpath = WORKER_ENV.get("PYTHONPATH", "")
WORKER_ENV["PYTHONPATH"] = str(PACKAGES) + (os.pathsep + old_pythonpath if old_pythonpath else "")
versions = subprocess.check_output(
    [sys.executable, "-c",
     "import cirq,numpy,sys; print(sys.version.split()[0], numpy.__version__, cirq.__version__)"],
    cwd=REPO, env=WORKER_ENV, text=True).strip()

def available_memory_bytes():
    rows = Path("/proc/meminfo").read_text(encoding="utf-8").splitlines()
    available = int(next(row.split()[1] for row in rows if row.startswith("MemAvailable:"))) * 1024
    for limit_path, usage_path in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                                   ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
                                    "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            limit = Path(limit_path).read_text().strip()
            if limit.isdecimal():
                available = min(available, max(0, int(limit) - int(Path(usage_path).read_text())))
        except (OSError, ValueError):
            pass
    return available

def process_rss_bytes(pid):
    try:
        status = Path(f"/proc/{pid}/status").read_text(encoding="utf-8")
        return int(next(row.split()[1] for row in status.splitlines()
                        if row.startswith("VmRSS:"))) * 1024
    except (OSError, StopIteration, ValueError):
        return 0

def terminate_worker(proc):
    if proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()

free_disk = shutil.disk_usage(RUN_DIR).free
print("SharkByte source commit:", SOURCE_COMMIT)
print("Notebook SHA-256:", NOTEBOOK_SHA256)
print("Python / NumPy / Cirq:", versions)
print("Host:", platform.platform(), platform.machine())
print(f"Available RAM: {available_memory_bytes() / GIB:.2f} GiB")
print(f"Free disk after checkout and dependencies: {free_disk / GIB:.2f} GiB")
print(f"Worker RSS stop: {WORKER_RSS_LIMIT / GIB:.2f} GiB")

In [ ]:
import resource

available_start = available_memory_bytes()
free_disk = shutil.disk_usage(RUN_DIR).free
candidates = []
for expanded in range(31, 25, -1):
    states = 1 << (expanded + 1)
    required = states * 24 + DISK_RESERVE
    source_path = REPO / "examples" / f"connected_ghz50_{states}.sb"
    if source_path.exists() and free_disk >= required:
        candidates.append((expanded, states, source_path, required))
print("Available RAM before computation:", f"{available_start / GIB:.2f} GiB")
print("Free disk for tier selection:", f"{free_disk / GIB:.2f} GiB")
if available_start < MIN_AVAILABLE_RAM:
    (RESULTS / "run-status.json").write_text(json.dumps({
        "status": "REFUSED", "reason": "Available RAM below 6 GiB preflight threshold.",
        "available_ram_bytes": available_start}, indent=2) + "\n", encoding="utf-8")
    candidates = []
candidates = candidates[:1]  # Use only the highest tier that passes preflight; lower-tier evidence is already archived.
if not candidates:
    print("No included tier fits the resource preflight; the run is REFUSED.")
    if not (RESULTS / "run-status.json").exists():
        (RESULTS / "run-status.json").write_text(json.dumps({
            "status": "REFUSED", "reason": "No tier fits free disk plus the 10 GiB reserve.",
            "free_disk_bytes": free_disk}, indent=2) + "\n", encoding="utf-8")
else:
    attempts = []
    selected_status = "NO_RUN"
    for expanded, states, source_path, required in candidates:
        tag = f"{states}"
        receipt = RESULTS / f"sharkbyte-connected-ghz50-{tag}-colab.json"
        progress = RESULTS / f"progress-{tag}.jsonl"
        log_path = RESULTS / f"worker-{tag}.log"
        scratch = SCRATCH / tag
        scratch.mkdir(parents=True, exist_ok=True)
        command = [sys.executable, str(REPO / "tools" / "run_connected_ghz50_materialization.py"),
                   "--source", str(source_path), "--output", str(receipt),
                   "--scratch-dir", str(scratch), "--progress-file", str(progress),
                   "--memory-budget-mib", str(int(WORKER_RSS_LIMIT / MIB)),
                   "--minimum-free-disk-gib", str(int(DISK_RESERVE / GIB)),
                   "--chunk-elements", str(CHUNK_ELEMENTS), "--max-seconds", "2400",
                   "--notebook-sha256", NOTEBOOK_SHA256]
        def set_worker_limits():
            resource.setrlimit(resource.RLIMIT_CPU, (WORKER_CPU_LIMIT, WORKER_CPU_LIMIT))
            resource.setrlimit(resource.RLIMIT_CORE, (0, 0))
        start = time.monotonic()
        sampled_peak = 0
        stop_reason = None
        with log_path.open("w", encoding="utf-8", newline="\n") as log_stream:
            proc = subprocess.Popen(command, cwd=REPO, env=WORKER_ENV,
                                    stdout=log_stream, stderr=subprocess.STDOUT,
                                    start_new_session=True, preexec_fn=set_worker_limits)
            try:
                while proc.poll() is None:
                    rss = process_rss_bytes(proc.pid)
                    sampled_peak = max(sampled_peak, rss)
                    elapsed = time.monotonic() - start
                    if rss > WORKER_RSS_LIMIT:
                        stop_reason = "PARENT_RSS_STOP"
                        terminate_worker(proc)
                        break
                    if elapsed > WORKER_TIMEOUT:
                        stop_reason = "PARENT_WALL_TIMEOUT"
                        terminate_worker(proc)
                        break
                    if int(elapsed) % 15 == 0:
                        latest = ""
                        if progress.exists():
                            try:
                                latest = progress.read_text(encoding="utf-8").splitlines()[-1]
                            except (OSError, IndexError):
                                pass
                        print(f"Running {states:,} states; elapsed={elapsed:.0f}s; "
                              f"sampled worker RSS={rss / GIB:.2f} GiB; {latest}", flush=True)
                    time.sleep(1)
                return_code = proc.wait()
            except KeyboardInterrupt:
                stop_reason = "NOTEBOOK_INTERRUPTED"
                terminate_worker(proc)
                return_code = proc.returncode if proc.returncode is not None else -signal.SIGINT
            finally:
                if proc.poll() is None:
                    terminate_worker(proc)
        elapsed = time.monotonic() - start
        if stop_reason:
            attempt = {"states": states, "status": stop_reason,
                       "elapsed_seconds": elapsed, "sampled_peak_worker_rss_bytes": sampled_peak,
                       "return_code": return_code}
            if receipt.exists():
                attempt["receipt_sha256"] = hashlib.sha256(receipt.read_bytes()).hexdigest()
            attempts.append(attempt)
            for p in scratch.glob("connected-ghz50-*"):
                try:
                    p.unlink()
                except OSError:
                    pass
            selected_status = stop_reason
            break
        receipt_status = None
        if receipt.exists():
            loaded = json.loads(receipt.read_text(encoding="utf-8"))
            receipt_status = loaded.get("status")
        attempt = {"states": states, "status": receipt_status or "WORKER_FAILED",
                   "elapsed_seconds": elapsed, "sampled_peak_worker_rss_bytes": sampled_peak,
                   "return_code": return_code}
        if receipt.exists():
            attempt["receipt_sha256"] = hashlib.sha256(receipt.read_bytes()).hexdigest()
        attempts.append(attempt)
        if return_code == 0 and receipt_status == "PASS":
            selected_status = "PASS"
            break
        if return_code == 3 and receipt_status == "RESOURCE_STOP":
            selected_status = receipt_status
            break
        if (return_code == 2 and receipt_status == "REFUSED" and receipt.exists()
                and "Insufficient free disk" in loaded.get("resource_refusal", "")):
            selected_status = receipt_status
            break
        selected_status = receipt_status or "WORKER_FAILED"
        break
    supervisor = {
        "format": "sharkbyte-colab-50q-supervisor-v1",
        "status": selected_status,
        "source_commit": SOURCE_COMMIT,
        "notebook_sha256": NOTEBOOK_SHA256,
        "host": platform.platform(), "architecture": platform.machine(),
        "python_numpy_cirq": versions,
        "available_ram_before_bytes": available_start,
        "free_disk_before_runs_bytes": free_disk,
        "worker_rss_stop_bytes": WORKER_RSS_LIMIT,
        "worker_timeout_seconds": WORKER_TIMEOUT,
        "attempts": attempts,
    }
    (RESULTS / "colab-supervisor.json").write_text(
        json.dumps(supervisor, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(supervisor, indent=2))
    if selected_status != "PASS":
        print("No PASS was recorded; inspect each receipt and worker log in the evidence bundle.")

In [ ]:
# Download the compact receipts, progress records, and logs (never the large temporary arrays).
archive_path = RUN_DIR / "sharkbyte-connected-ghz50-2p32-retry-results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.iterdir()):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print("Evidence bundle:", archive_path)
from google.colab import files
files.download(str(archive_path))